## Homework 4: Evaluation

This is the personal implementation of the homework 4 of [LLM Zoomcamp 2026 Cohort](https://github.com/DataTalksClub/llm-zoomcamp/tree/main).In this homework we generate a ground truth dataset and use it to evaluate search. It reuses the same chunks and the same search functions from homework 2. The course repository is organized by modules and each module is a top-level folder with a `lessons/` subfolder of numbered markdown pages:

```python
01-agentic-rag/
└── lessons/
    ├── 01-intro.md
    ├── 02-environment.md
    ├── ...
    └── 16-other-frameworks.md
```

Each lesson page is a single markdown file, which are exactly the data to be fetched from GitHub and to be used as the knowledge base. 

Environment required both the vector search preparation as in module 2 and also evaluation module specific libraries:

```python
cd module4_homework
uv init --no-workspace
uv add onnxruntime tokenizers numpy tqdm minsearch gitsource
uv add --dev huggingface-hub jupyter

uv add openai pydantic python-dotenv pandas

PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed
wget $PREFIX/download.py
wget $PREFIX/embedder.py

uv run python download.py
uv run python -m ipykernel install --user --name llm-zoomcamp-onnx --display-name "llm-zoomcamp-onnx"
```

After creating the new kernel, check if it is created properly:
```python
jupyter kernelspec list
```
Then if it appears in the list, but does not appear as Jupyter Kernel to change into, reload the window.

In [39]:
# pull the lesson pages from the course repository
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

print(f"Total lesson pages: {len(documents)}")

Total lesson pages: 72


In [40]:
# sample document
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [4]:
PREFIX = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"

!wget {PREFIX}/01-agentic-rag/code/rag_helper.py
!wget {PREFIX}/04-evaluation/code/evaluation_utils.py

--2026-07-13 16:09:09--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 

200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py’

rag_helper.py       100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-07-13 16:09:09 (40.7 MB/s) - ‘rag_helper.py’ saved [2134/2134]

--2026-07-13 16:09:09--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/code/evaluation_utils.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3073 (3.0K) [text/plain]
Saving to: ‘evaluation_utils.py’

evaluation_utils.py 100%[===================>]   3.00K  --.-KB/s    in 0s      

2026-07-13 16:09:10 (45.6 MB/s) - ‘evaluation_utils.py’ saved [3073/3073]



In [41]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

# adapt model instructions to the context of the course
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [42]:
# make sure .env file is loaded (working directory is important here)
from dotenv import load_dotenv
print(load_dotenv())

# check whether the OpenAI client works
from openai import OpenAI
openai_client = OpenAI()

True


In [43]:
from evaluation_utils import llm_structured
import json

# create a function to create ground truth records for a given document
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "filename ": doc["filename"]
        })

    return results, usage

### Q1. Generating questions

When generating questions for the first 3 lesson pages, what is the average number of input tokens across these 3 calls?

In [44]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

# try for the first 3 documents
for doc in tqdm(documents[:3]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/3 [00:00<?, ?it/s]

In [45]:
avg_input = sum(u.input_tokens for u in usages) / len(usages)
print(f"Average number of input tokens in {len(usages)} calls: {avg_input:.1f}")

Average number of input tokens in 3 calls: 1353.0


#### Chunking and Building Search Index

In [26]:
PREFIX = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"
!wget {PREFIX}/cohorts/2026/04-evaluation/ground-truth.csv

--2026-07-13 18:36:50--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/04-evaluation/ground-truth.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 48627 (47K) [text/plain]
Saving to: ‘ground-truth.csv’

ground-truth.csv    100%[===================>]  47.49K  --.-KB/s    in 0.002s  

2026-07-13 18:36:50 (24.1 MB/s) - ‘ground-truth.csv’ saved [48627/48627]



In [36]:
PREFIX = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed"
!wget {PREFIX}/download.py
!wget {PREFIX}/embedder.py

--2026-07-13 19:12:33--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/download.py


Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1376 (1.3K) [text/plain]
Saving to: ‘download.py’

download.py         100%[===================>]   1.34K  --.-KB/s    in 0s      

2026-07-13 19:12:34 (82.6 MB/s) - ‘download.py’ saved [1376/1376]

--2026-07-13 19:12:34--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/embedder.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1520 (1.5K) [text/plain]
Saving to: ‘embedder.py.1’

embedder.py.1       100%[===================>]   1.48K  --.-KB

In [46]:
# get the full ground truth from the loaded file
import pandas as pd

df_ground_truth = pd.read_csv("ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

print(len(ground_truth))
print(ground_truth[0])

360
{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?", 'filename': '01-agentic-rag/lessons/01-intro.md'}


In [47]:
# create chunks from documents
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

print(f"Total number of chunks: {len(chunks)}")
chunks[0]

Total number of chunks: 295


{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [48]:
# text search index
from minsearch import Index

# define a  function to index with minsearch
def build_index(documents):
    index = Index(
        text_fields=["content"],
        keyword_fields=["filename"]
    )
    index.fit(documents)
    return index

# create the chunks index
text_index = build_index(chunks)

In [49]:
# text search
def text_search(query, num_results=5):
    return text_index.search(
        query,
        num_results=num_results
    )

In [50]:
# vector search index
from embedder import Embedder
from minsearch import VectorSearch
import numpy as np

embed = Embedder()

contents = [chunk["content"] for chunk in chunks]

# embed the contents
vectors = embed.encode_batch(contents)

# turn vectors into a matrix where rows are documents (vectors), and cols are dimensions of them
X = np.array(vectors)
print(f"There are {X.shape[0]} vectors with {X.shape[1]} dimensions.")

# index the chunks and vectors
vindex = VectorSearch()
vindex.fit(X, chunks)

There are 295 vectors with 384 dimensions.


In [51]:
def vector_search(query, num_results=5):
    query_vector = embed.encode(query)

    results = vindex.search(
        query_vector,
        num_results=num_results
    )

    return results

In [52]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

# hybrid searech function that combines text and vector search results using RRF
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

### Q2. First result with text search

In [53]:
# take the first question from the ground truth to test the search functions
q = ground_truth[0]["question"]

# perform text search
results = text_search(query=q)

# filename of the first result
print(f"File name of the 1st result: {results[0]['filename']}")

File name of the 1st result: 01-agentic-rag/lessons/03-rag.md


### Q3. First result with vector search

In [54]:
# perform vector search
results = vector_search(query=q)

# filename of the first result
print(f"File name of the 1st result: {results[0]['filename']}")

File name of the 1st result: 01-agentic-rag/lessons/01-intro.md


#### Evaluation Metrics

In [58]:
# make relevance search generic
def compute_relevance(query, search_function):
    filename = query["filename"]
    results = search_function(query=query["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == filename))

    return relevance

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [60]:
# hit rate as a function
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

# mrr as a function
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

# combining metrics into a single function
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }


### Q4. Evaluating text search

In [ ]:
# text search evaluation
print(f"Hit rate: {evaluate(ground_truth, text_search)["hit_rate"]:.2f}")

  0%|          | 0/360 [00:00<?, ?it/s]

Hit rate: 0.76


### Q5. Evaluating vector search

In [64]:
# vector search evaluation
print(f"MRR: {evaluate(ground_truth, vector_search)["mrr"]:.2f}")

  0%|          | 0/360 [00:00<?, ?it/s]

MRR: 0.55


### Q6. Tuning hybrid search

In [65]:
# evaluate several k values
for k in [1, 50, 100, 200]:
    result = evaluate(
        ground_truth,
        lambda query, k=k: hybrid_search(query, k=k)
    )
    print(f"k={k}: {result}")

  0%|          | 0/360 [00:00<?, ?it/s]

k=1: {'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444449}


  0%|          | 0/360 [00:00<?, ?it/s]

k=50: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}


  0%|          | 0/360 [00:00<?, ?it/s]

k=100: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}


  0%|          | 0/360 [00:00<?, ?it/s]

k=200: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}
